# GPU fully online end-to-end AMC inference: 2022train20

Each test signal is read from its raw `.tim` IQ file and independently follows BOI detection/frequency shifting, three FFT representations, LR55 coarse classification, and—only for the QPSK-like route—blind $R_s$/CFO estimation, $11\times24$ CC construction, $11\times15$ cropping, and CAPNet fine classification. Ground-truth labels are used only for the 2018 negative-SNR exclusion rule and final evaluation.

The two evaluated splits are: (1) CSPB.ML.2022 batches 25–28 and (2) the complete nonnegative-SNR CSPB.ML.2018 dataset. Results are checkpointed so a long run can resume safely.

This version uses `helper_function.fully_online_pipeline_inference_gpu` to run BOI/FFT feature extraction, LR55 scoring, CC construction, and CAP inference with PyTorch tensors on the selected device where practical. File I/O and CSV reporting remain CPU-side.

In [1]:
from pathlib import Path
import sys


def _find_helper_project_dir(start=Path.cwd()):
    start = Path(start).resolve()
    search_roots = [start, *start.parents]
    for root in search_roots:
        if (root / "helper_function" / "__init__.py").exists():
            return root
    for root in search_roots[:3]:
        matches = list(root.glob("**/boi_shift_fft3_cfpattern_spl/helper_function/__init__.py"))
        if matches:
            return matches[0].parent.parent
    raise FileNotFoundError("Could not find helper_function from the current Jupyter working directory.")


PROJECT_DIR = _find_helper_project_dir()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from helper_function.fully_online_pipeline_inference_gpu import (
    FullyOnlinePipelineGPU,
    make_default_config,
    print_and_plot_cm,
)

BASE_DIR = PROJECT_DIR
DATASET_ROOT = (BASE_DIR / '../../../../../../CSPB.ML_dataset').resolve()
MODEL_DIR = BASE_DIR / 'fully_online_pipeline_models'
RESULT_DIR = BASE_DIR / 'fully_online_inference_results_gpu' / '2022train20'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

CFG = make_default_config(DATASET_ROOT)
pipeline = FullyOnlinePipelineGPU(
    dataset_root=DATASET_ROOT,
    model_dir=MODEL_DIR,
    split_mode='2022train20',
    cfg=CFG,
    device='cuda:1',
)

print('Dataset root:', DATASET_ROOT)
print('LR model:', pipeline.lr_path)
print('CAP model:', pipeline.cap_path)
print('Device:', pipeline.device)
print('GPU helper:', pipeline.__class__.__name__)
print('LR feature count:', len(pipeline.lr_feature_names))

/home/ruilin/miniconda3/envs/fpt_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset root: /home/ruilin/SPL_Surreal/CSPB.ML_dataset
LR model: /home/ruilin/SPL_Surreal/Surreal_source/chad-reproduce/chad-reproduce-sensors/BOI+SSCA+CC-Blind-generateselectedBOI+SSCA-metadata/BOI+FFT-2018+2022-generate/boi_shift_fft3_cfpattern_spl/fully_online_pipeline_models/lr55_2022train20.joblib
CAP model: /home/ruilin/SPL_Surreal/Surreal_source/chad-reproduce/chad-reproduce-sensors/BOI+SSCA+CC-Blind-generateselectedBOI+SSCA-metadata/BOI+FFT-2018+2022-generate/boi_shift_fft3_cfpattern_spl/fully_online_pipeline_models/capnet_2022train20.pt
Device: cuda:1
GPU helper: FullyOnlinePipelineGPU
LR feature count: 55


## Confusion matrix 1: 2022 held-out batches 25–28

In [ ]:
results_2022 = pipeline.run_dataset(
    year=2022,
    batch_ids=range(25, 29),
    output_csv=RESULT_DIR / '2022_batches25_28_predictions.csv',
    resume=True,
    checkpoint_every=100,
    fail_fast=True,
)
cm_2022 = print_and_plot_cm(results_2022, '2022train20: CSPB.ML.2022 batches 25–28')

2022train20 GPU: 2022:   9%|▉         | 1494/16000 [9:27:18<80:55:01, 20.08s/signal] 

## Confusion matrix 2: complete nonnegative-SNR 2018 cross-domain test set

In [ ]:
results_2018 = pipeline.run_dataset(
    year=2018,
    batch_ids=range(1, 29),
    output_csv=RESULT_DIR / '2018_nonnegative_snr_predictions.csv',
    resume=True,
    checkpoint_every=100,
    fail_fast=True,
)
cm_2018 = print_and_plot_cm(results_2018, '2022train20: CSPB.ML.2018 nonnegative-SNR cross-domain test')

## Confusion Matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from sklearn.metrics import confusion_matrix

plt.rcParams.update({
    "font.family": "STIXGeneral",
    "mathtext.fontset": "stix",
})

PLOT_CLASS_ORDER = [
    "BPSK",
    "QPSK",
    "8PSK",
    "DQPSK",
    "MSK",
    "16QAM",
    "64QAM",
    "256QAM",
]

def row_normalize_cm(cm):
    cm = np.asarray(cm, dtype=np.float64)
    row_sum = cm.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0
    return cm / row_sum * 100.0

def cm_acc_from_results(results, class_order=PLOT_CLASS_ORDER):
    df = results.copy()

    if "pred_label" not in df.columns or "true_label" not in df.columns:
        raise ValueError("results must contain true_label and pred_label columns.")

    if df["pred_label"].astype(str).eq("ERROR").any():
        n_err = int(df["pred_label"].astype(str).eq("ERROR").sum())
        raise RuntimeError(f"Found {n_err} ERROR samples. Inspect results before plotting.")

    if "error" in df.columns:
        bad = df["error"].fillna("").astype(str).str.len() > 0
        if bad.any():
            raise RuntimeError(f"Found {int(bad.sum())} non-empty error rows. Inspect results before plotting.")

    y_true = df["true_label"].astype(str).str.strip()
    y_pred = df["pred_label"].astype(str).str.strip()

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=class_order,
    )

    acc = np.mean(y_true.to_numpy() == y_pred.to_numpy()) * 100.0
    return cm, acc

def plot_combined_confusion(
    cm_2018,
    cm_2022,
    acc_2018,
    acc_2022,
    class_names=PLOT_CLASS_ORDER,
    save_path=None,
):
    cm_2018_pct = row_normalize_cm(cm_2018)
    cm_2022_pct = row_normalize_cm(cm_2022)

    n_classes = len(class_names)
    background = np.eye(n_classes)

    cmap = ListedColormap([
        "#FFFFFF",
        "#CFE8F3",
    ])

    fig, ax = plt.subplots(figsize=(16, 14))

    ax.imshow(
        background,
        cmap=cmap,
        vmin=0,
        vmax=1,
        interpolation="nearest",
        aspect="equal",
    )

    ax.set_xticks(np.arange(-0.5, n_classes, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_classes, 1), minor=True)

    ax.grid(
        which="minor",
        color="#78909C",
        linestyle="-",
        linewidth=1.4,
    )
    ax.tick_params(which="minor", bottom=False, left=False)

    ax.set_xticks(np.arange(n_classes))
    ax.set_yticks(np.arange(n_classes))

    ax.set_xticklabels(
        class_names,
        rotation=52,
        ha="right",
        fontsize=28,
    )
    ax.set_yticklabels(
        class_names,
        fontsize=28,
    )

    ax.set_xlabel(
        "Predicted Class",
        fontsize=34,
        labelpad=18,
    )
    ax.set_ylabel(
        "True Class",
        fontsize=34,
        labelpad=18,
    )

    for i in range(n_classes):
        for j in range(n_classes):
            value_2018 = cm_2018_pct[i, j]
            value_2022 = cm_2022_pct[i, j]

            if round(value_2018, 1) == 0.0 and round(value_2022, 1) == 0.0:
                continue

            ax.text(
                j,
                i - 0.14,
                f"{value_2018:.1f}%",
                ha="center",
                va="center",
                color="#D32F2F",
                fontsize=30,
                fontweight="bold",
            )

            ax.text(
                j,
                i + 0.18,
                f"{value_2022:.1f}%",
                ha="center",
                va="center",
                color="#2E7D32",
                fontsize=30,
                fontweight="bold",
            )

    legend_handles = [
        Line2D(
            [0], [0],
            color="#D32F2F",
            linewidth=5,
            label=f"2018 test: Acc = {acc_2018:.2f}%",
        ),
        Line2D(
            [0], [0],
            color="#2E7D32",
            linewidth=5,
            label=f"2022 test: Acc = {acc_2022:.2f}%",
        ),
    ]

    ax.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=2,
        frameon=True,
        fontsize=25,
        handlelength=2.2,
        columnspacing=2.5,
    )

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.tick_params(axis="both", which="major", length=0, pad=12)

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(
            save_path,
            dpi=600,
            bbox_inches="tight",
            facecolor="white",
        )

    plt.show()


cm_2018, acc_2018 = cm_acc_from_results(results_2018)
cm_2022, acc_2022 = cm_acc_from_results(results_2022)

print(f"2018 test: N = {len(results_2018):,}, Acc = {acc_2018:.2f}%")
print(f"2022 test: N = {len(results_2022):,}, Acc = {acc_2022:.2f}%")

plot_combined_confusion(
    cm_2018=cm_2018,
    cm_2022=cm_2022,
    acc_2018=acc_2018,
    acc_2022=acc_2022,
    class_names=PLOT_CLASS_ORDER,
    save_path=None,
    # save_path="combined_confusion_matrix.pdf",
)